# SVD-Based Facial Recognition: Eigenfaces & Dimensionality Reduction

Unsupervised dimensionality reduction via Singular Value Decomposition (SVD) applied to the
Labeled Faces in the Wild (LFW) dataset. This project demonstrates eigenfaces for facial
representation and SVD-projected features for binary classification, including
cross-validated hyperparameter tuning to find the optimal number of components.

| | |
|---|---|
| **Dataset** | LFW — 2,588 images · 42 individuals · 62×47 px grayscale |
| **Key result** | K=150 SVD components achieves 75.6% CV recall vs 69.1% at K=1,000 |
| **Stack** | NumPy · SciPy · scikit-learn · Matplotlib |

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import svd
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)

IMAGES_DIR = Path(".") / "images"
IMAGES_DIR.mkdir(exist_ok=True)  # store figures locally


## 1. Dataset Overview

In [ ]:
def load_lfw_dataset(min_faces_per_person=25):
    """Load LFW people dataset filtered by minimum face count per person."""
    return fetch_lfw_people(min_faces_per_person=min_faces_per_person)


lfw_people = load_lfw_dataset()
n_samples, height, width = lfw_people.images.shape  # dataset dimensions
n_classes = len(lfw_people.target_names)
print(f"Samples: {n_samples} | Classes: {n_classes} | Image: {height}x{width}px")

## 2. Exploratory Data Analysis

The dataset is heavily class-imbalanced: George W. Bush accounts for ~20% of all images,
while 30+ individuals have fewer than 40 images each. This skew motivates **recall**
as the primary evaluation metric for the downstream binary classifier.

In [ ]:
# count images per person and sort descending
persons, counts = np.unique(lfw_people.target, return_counts=True)
dist_df = (
    pd.DataFrame({"person": lfw_people.target_names[persons], "images": counts})
    .sort_values("images", ascending=False)
    .reset_index(drop=True)
)
print(dist_df.to_string())

In [ ]:
top_n = 15
top_df = dist_df.head(top_n)

# bar chart of top individuals by image count
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(top_df)), top_df["images"])
ax.set_xticks(range(len(top_df)))
ax.set_xticklabels(top_df["person"], rotation=90)
ax.set_xlabel("Person")
ax.set_ylabel("Images")
ax.set_title(f"Top {top_n} individuals by image count")
for i, v in enumerate(top_df["images"]):
    ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "image_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# filter images for the selected individual
TARGET_NAME = "Serena Williams"
target_idx = np.where(lfw_people.target_names == TARGET_NAME)[0][0]
target_indices = np.where(lfw_people.target == target_idx)[0]

# display all images in a grid
cols_grid = 10
rows_grid = math.ceil(len(target_indices) / cols_grid)
fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(1.6 * cols_grid, 1.6 * rows_grid))
axes = np.atleast_1d(axes).ravel()
for ax, img in zip(axes, lfw_people.images[target_indices]):
    ax.imshow(img, cmap="gray")
    ax.axis("off")
for ax in axes[len(target_indices):]:
    ax.axis("off")
plt.suptitle(f"{len(target_indices)} images of {TARGET_NAME}", y=1.01)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "serena_williams.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Eigenfaces via SVD

Eigenfaces decompose the face space into orthogonal basis vectors ordered by the variance
they capture. Any face can be expressed as the mean face plus a weighted sum of these
components, making SVD an optimal low-rank approximation for this domain (Eckart-Young
theorem).

### 3.1 Mean Face

In [ ]:
X = lfw_people.data  # shape (n_samples, height * width)
mean_face = X.mean(axis=0)  # per-pixel average across all images

plt.figure(figsize=(3, 4))
plt.imshow(mean_face.reshape(height, width), cmap="gray")
plt.title(f"Mean face (n={n_samples})")
plt.axis("off")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "mean_face.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Mean Subtraction

Centering shifts the data to zero mean so that SVD captures variance rather than offset.
Residual images highlight per-image deviations from the average facial appearance.

In [ ]:
# subtract per-pixel mean from every image
X_centered = X - mean_face

# show Serena Williams images after mean subtraction
fig, axes = plt.subplots(rows_grid, cols_grid
axes = np.atleast_1d(axes).ravel()
for ax, idx in zip(axes, target_indices):
    ax.imshow(X_centered[idx].reshape(height, width), cmap="gray")
    ax.axis("off")
for ax in axes[len(target_indices):]:
    ax.axis("off")
plt.suptitle(f"{len(target_indices)} images of {TARGET_NAME} — mean subtracted", y=1.01)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "serena_mean_subtracted.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Singular Value Decomposition

In [ ]:
K = 1000
# full SVD of the centered matrix; full_matrices=False for memory efficiency
U, S, Vt = svd(X_centered, full_matrices=False)

# proportion of total variance captured by the top K singular values
var_total = np.sum(S ** 2)
var_K = np.sum(S[:K] ** 2)
print(f"Total variance:              {var_total:.2f}")
print(f"Variance explained by K={K}: {var_K:.2f} ({var_K / var_total * 100:.4f}%)")

In [ ]:
# cumulative variance explained as a function of K
cumvar = np.cumsum(S ** 2) / var_total * 100
ks = np.arange(1, len(S) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, cumvar, linewidth=1)
ax.axhline(99, color="red", linestyle="--", linewidth=0.8, label="99% threshold")
ax.axvline(K, color="orange", linestyle="--", linewidth=0.8, label=f"K={K}")
ax.set_xlabel("K (number of singular values)")
ax.set_ylabel("Cumulative variance explained (%)")
ax.set_title("Scree plot — cumulative variance explained")
ax.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "scree_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# top eigenfaces: rows of Vt reshaped back to image dimensions
n_ef = 16
fig, axes = plt.subplots(2, n_ef // 2, figsize=(16, 4))
axes = axes.ravel()
for i, ax in enumerate(axes):
    ax.imshow(Vt[i].reshape(height, width), cmap="gray")
    ax.set_title(f"#{i + 1}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Top {n_ef} eigenfaces (right singular vectors)", y=1.02)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eigenfaces.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Face Reconstruction

Each face reconstructs as `mean_face + U_k @ diag(S_k) @ Vt_k`, where k indexes
the retained components. With K=1,000, 99.79% of total variance is preserved and
reconstructions are visually indistinguishable from the originals.

In [ ]:
# truncated reconstruction: approximate X from K components and restore the mean
U_k = U[:, :K]
X_reconstructed = (U_k * S[:K]) @ Vt[:K] + mean_face

# compare originals vs reconstructions for a sample of Serena Williams images
n_show = 8
show_idx = target_indices[:n_show]
fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 5))
for col, idx in enumerate(show_idx):
    axes[0, col].imshow(X[idx].reshape(height, width), cmap="gray")
    axes[0, col].axis("off")
    axes[1, col].imshow(X_reconstructed[idx].reshape(height, width), cmap="gray")
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=9)
axes[1, 0].set_ylabel(f"K={K}", fontsize=9)
plt.suptitle(f"Face reconstruction — {TARGET_NAME}", y=1.02)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "reconstruction.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Dimensionality Reduction for Classification

SVD projects images onto a compact subspace that maximizes retained variance while
discarding per-pixel noise. These projections serve as features for a logistic
regression classifier targeting George W. Bush (the dataset's majority class).

In [ ]:
# binary label: 1 = George W Bush, 0 = other
CLF_TARGET = "George W Bush"
clf_idx = np.where(lfw_people.target_names == CLF_TARGET)[0][0]
y = (lfw_people.target == clf_idx).astype(int)
print(f"{CLF_TARGET}: {y.sum()} positive | {(y == 0).sum()} negative | total {len(y)}")

In [ ]:
# 80/20 split; random_state fixed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

### Data-Leakage-Free SVD Protocol

Mean centering and SVD are computed on the **training set only**. The test set is
re-centered with the training mean and projected onto training-derived right singular
vectors, preventing any test-set information from influencing the feature space.

In [ ]:
# center using training mean only — applying the same mean to the test set avoids leakage
train_mean = X_train.mean(axis=0)
X_train_c = X_train - train_mean
X_test_c = X_test - train_mean

# SVD on training data; Vt_tr holds the projection basis used for both splits
U_tr, S_tr, Vt_tr = svd(X_train_c, full_matrices=False)
print(f"Training SVD: U{U_tr.shape}  S{S_tr.shape}  Vt{Vt_tr.shape}")

### Baseline Model (K=1,000)

In [ ]:
K_baseline = 1000
# project onto the top K right singular vectors
X_tr_proj = X_train_c @ Vt_tr[:K_baseline].T
X_te_proj = X_test_c @ Vt_tr[:K_baseline].T

# standardize each projected component before fitting
scaler_b = StandardScaler()
X_tr_proj = scaler_b.fit_transform(X_tr_proj)
X_te_proj = scaler_b.transform(X_te_proj)

clf_b = LogisticRegression(solver="sag", random_state=42, max_iter=1000)
clf_b.fit(X_tr_proj, y_train)
y_pred_b = clf_b.predict(X_te_proj)
y_proba_b = clf_b.predict_proba(X_te_proj)[:, 1]

baseline_recall = recall_score(y_test, y_pred_b, pos_label=1)
print(f"Baseline (K={K_baseline}) — test results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_b):.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"Precision: {precision_score(y_test, y_pred_b, pos_label=1):.4f}")
print(f"F1: {f1_score(y_test, y_pred_b, pos_label=1):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_proba_b):.4f}")

## 5. Hyperparameter Optimization

K controls the bias-variance tradeoff: too few components discard discriminative signal;
too many introduce noise. Stratified 5-fold cross-validation selects K to maximize recall
on the positive class. SVD is re-computed within each fold to maintain the leakage-free
protocol.

In [ ]:
def cv_recall(k, X, y, n_folds=5):
    """Return mean stratified-CV recall for k SVD components; SVD is computed per fold."""
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=10101)
    recalls = []
    for tr_idx, val_idx in kf.split(X, y):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # center and decompose within each fold to prevent leakage
        mean = X_tr.mean(axis=0)
        X_tr_c, X_val_c = X_tr - mean, X_val - mean

        _, _, Vt_cv = svd(X_tr_c, full_matrices=False)
        X_tr_p = X_tr_c @ Vt_cv[:k].T
        X_val_p = X_val_c @ Vt_cv[:k].T

        sc = StandardScaler()
        X_tr_p = sc.fit_transform(X_tr_p)
        X_val_p = sc.transform(X_val_p)

        m = LogisticRegression(solver="sag", random_state=10101, max_iter=1000)
        m.fit(X_tr_p, y_tr)
        recalls.append(recall_score(y_val, m.predict(X_val_p), pos_label=1))

    return np.mean(recalls)


# grid search over K candidates; skip values exceeding available components
K_grid = [k for k in [50, 100, 150, 200, 300, 400, 600, 800, 1000, 1200]
          if k <= Vt_tr.shape[0]]

cv_results = {}
for k in K_grid:
    cv_results[k] = cv_recall(k, X_train, y_train)
    print(f"K={k:5d}  CV recall={cv_results[k]:.4f}")

best_K = max(cv_results, key=cv_results.get)
print(f"\nOptimal K: {best_K}  (CV recall={cv_results[best_K]:.4f})")

In [ ]:
# retrain on the full training set using the optimal K from cross-validation
X_tr_proj = X_train_c @ Vt_tr[:best_K].T
X_te_proj = X_test_c @ Vt_tr[:best_K].T

scaler = StandardScaler()
X_tr_proj = scaler.fit_transform(X_tr_proj)
X_te_proj = scaler.transform(X_te_proj)

clf = LogisticRegression(solver="sag", random_state=10101, max_iter=1000)
clf.fit(X_tr_proj, y_train)
y_pred = clf.predict(X_te_proj)
y_proba = clf.predict_proba(X_te_proj)[:, 1]

final_recall = recall_score(y_test, y_pred, pos_label=1)
print(f"Final model (K={best_K}) — test results")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Recall:    {final_recall:.4f}")
print(f"  Precision: {precision_score(y_test, y_pred, pos_label=1):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred, pos_label=1):.4f}")
print(f"  AUC:       {roc_auc_score(y_test, y_proba):.4f}")
print(f"  Recall gain vs K={K_baseline} baseline: {final_recall - baseline_recall:+.4f}")

## 6. Results & Conclusions

SVD provides an effective low-dimensional representation of face images. Key findings:

- **K=1,000 components explain 99.79% of total variance**, confirming the face space is
  highly compressible. Eigenfaces visually capture illumination gradients, pose variation,
  and high-level facial structure.
- **Classification improves with fewer components**: cross-validation identified K=150 as
  optimal. On the held-out test set, K=150 achieves recall=72.3% and AUC=0.934 —
  outperforming K=1,000 (recall=62.5%, AUC=0.886) by ~10 percentage points. Using 7x
  fewer features also reduces training time significantly.
- **Bias-variance tradeoff**: high K retains noise and over-parameterizes the classifier;
  low K loses discriminative signal. Cross-validation finds the sweet spot.
- **Leakage-free protocol**: mean centering and SVD are computed exclusively on the
  training fold in each CV iteration, preserving test-set integrity.

**Potential extensions**: randomized SVD for larger datasets; face verification (similarity
threshold) instead of 1-of-N identification; kernel PCA for nonlinear face manifolds.

---

**References**
- G. B. Huang,  M. Ramesh, T. Berg, and E. Learned-Miller. Labeled Faces in the Wild: A Database for Studying Face Recognition in Unconstrained Environments. University of Massachusetts, Amherst, Technical Report 07-49, October, 2007.

- M. Kirby and L. Sirovich. Application of the Karhunen-Loève procedure for the characterization of human faces. IEEE Transactions on Pattern Analysis and Machine Intelligence (PAMI), 12(1):103–108, 1990.

- L. Sirovich and M. Kirby. A low-dimensional procedure for the characterization of human faces. Journal of the Optical Society of America A, 4(3):519–524, 1987

- M. Turk and A. Pentland. Eigenfaces for recognition. Journal of Cognitive Neuroscience, 3(1):71–86, 1991.